# llm-from-scratch — train on a free GPU

A from-scratch GPT for cybersecurity + software development.

**Before running:** Runtime ▸ Change runtime type ▸ **T4 GPU**, then Runtime ▸ Run all.

- Milestone 2: `--preset gpu` (~10M params, RFC corpus).
- Milestone 3: `--preset gpu_large` (~32M params, RFC **+ NIST** corpus). Used below.

Repo: https://github.com/Canadasun/llm-from-scratch

In [ ]:
# 1. Get the code (PyTorch is already installed on Colab/Kaggle).
![ -d llm-from-scratch ] || git clone https://github.com/Canadasun/llm-from-scratch
%cd llm-from-scratch
import torch
print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
# 2. Build the Milestone-3 corpus: curated IETF RFCs + NIST SP 800 publications.
#    PyMuPDF is needed to extract text from the NIST PDFs. Secrets are scrubbed
#    by default. This downloads ~150 RFCs + ~27 NIST PDFs the first time
#    (cached afterwards); allow a few minutes.
!pip install -q pymupdf
!python data/build_corpus.py

In [ ]:
# 3. Train the large preset (~32M params, BPE vocab 2048, block 512).
#    On a T4 this is roughly 2 hours for 6000 iters; the first run also spends
#    a few minutes building the BPE tokenizer, then caches it.
#    To shorten: add --max-iters 3000.  To use the smaller M2 model: --preset gpu.
!python train.py --preset gpu_large

In [ ]:
# 4. Generate text from your trained model.
!python sample.py --prompt "Zero trust architecture " --tokens 400

In [ ]:
# 5. Measure it: intrinsic metrics vs a random baseline + capability tasks.
!python -m evals.run

In [ ]:
# 6. Download the trained checkpoint so you keep it.
try:
    from google.colab import files
    files.download('checkpoints/ckpt.pt')
except Exception as e:
    print('Not on Colab or no file:', e)